# Bayesian Optimization with Intelligent Bias

## Overview

Bayesian Optimization is particularly suitable for optimizing expensive black-box functions. In NSGABLACK, we integrate it with the bias system for enhanced performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize

print("Bayesian Optimization Tutorial")

## Gaussian Process Model

In [ ]:
class GaussianProcess:
    def __init__(self, kernel='RBF', alpha=1e-10):
        self.kernel = kernel
        self.alpha = alpha
        self.X_train = None
        self.y_train = None
        self.K_inv = None
    
    def rbf_kernel(self, x1, x2, length_scale=1.0):
        sqdist = np.sum(x1**2, 1).reshape(-1, 1) + np.sum(x2**2, 1) - 2 * np.dot(x1, x2.T)
        return np.exp(-0.5 / length_scale**2 * sqdist)
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        K = self.rbf_kernel(X, X) + self.alpha * np.eye(len(X))
        self.K_inv = np.linalg.inv(K)
    
    def predict(self, X_test):
        K_star = self.rbf_kernel(self.X_train, X_test)
        K_star_star = self.rbf_kernel(X_test, X_test)
        
        mu = K_star.T @ self.K_inv @ self.y_train
        sigma = K_star_star - K_star.T @ self.K_inv @ K_star
        
        return mu.flatten(), np.diag(sigma)

# Test Gaussian Process
gp = GaussianProcess()
X_train = np.array([[1], [2], [3], [4]])
y_train = np.array([2, 3, 2.5, 4])

gp.fit(X_train, y_train)
X_test = np.linspace(0, 5, 100).reshape(-1, 1)
mu, sigma = gp.predict(X_test)

print(f"GP fitted. Predictions at X=2.5: {mu[50]:.3f} +/- {sigma[50]:.3f}")

## Acquisition Functions

In [ ]:
def expected_improvement(mu, sigma, y_best, xi=0.01):
    with np.errstate(divide='warn'):
        imp = mu - y_best - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei

def upper_confidence_bound(mu, sigma, kappa=2.0):
    return mu + kappa * sigma

def probability_of_improvement(mu, sigma, y_best, xi=0.01):
    with np.errstate(divide='warn'):
        imp = mu - y_best - xi
        Z = imp / sigma
        pi = norm.cdf(Z)
        pi[sigma == 0.0] = 0.0
    return pi

# Test acquisition functions
mu_test = np.array([1, 2, 3])
sigma_test = np.array([0.5, 1.0, 0.2])
y_best = 2.5

ei = expected_improvement(mu_test, sigma_test, y_best)
ucb = upper_confidence_bound(mu_test, sigma_test)
pi = probability_of_improvement(mu_test, sigma_test, y_best)

print(f"EI: {ei}")
print(f"UCB: {ucb}")
print(f"PI: {pi}")

## Bayesian Optimization with Bias

In [ ]:
class BayesianOptimizationWithBias:
    def __init__(self, f, bounds, bias_strength=0.1):
        self.f = f
        self.bounds = bounds
        self.bias_strength = bias_strength
        self.gp = GaussianProcess()
        self.X_observed = []
        self.y_observed = []
        
    def initialize(self, n_init=5):
        for _ in range(n_init):
            x = np.random.uniform(self.bounds[0], self.bounds[1])
            y = self.f(x)
            self.X_observed.append([x])
            self.y_observed.append(y)
        
        self.X_observed = np.array(self.X_observed)
        self.y_observed = np.array(self.y_observed)
    
    def propose_location(self, acquisition='EI'):
        X_test = np.linspace(self.bounds[0], self.bounds[1], 1000).reshape(-1, 1)
        mu, sigma = self.gp.predict(X_test)
        
        if acquisition == 'EI':
            scores = expected_improvement(mu, sigma, np.min(self.y_observed))
        elif acquisition == 'UCB':
            scores = upper_confidence_bound(mu, sigma)
        else:
            scores = probability_of_improvement(mu, sigma, np.min(self.y_observed))
        
        # Apply bias toward good regions
        if len(self.X_observed) > 0:
            for i, x in enumerate(X_test):
                for x_obs in self.X_observed:
                    dist = abs(x[0] - x_obs[0])
                    if dist < 0.5 and self.y_observed[np.where(self.X_observed == x_obs)[0][0]] < np.mean(self.y_observed):
                        scores[i] *= (1 + self.bias_strength)
        
        return X_test[np.argmax(scores)][0]
    
    def optimize(self, n_iter=50):
        self.initialize()
        
        for iteration in range(n_iter):
            self.gp.fit(self.X_observed, self.y_observed)
            x_next = self.propose_location()
            y_next = self.f(x_next)
            
            self.X_observed = np.vstack([self.X_observed, [x_next]])
            self.y_observed = np.append(self.y_observed, y_next)
            
            if iteration % 10 == 0:
                print(f"Iteration {iteration}: Best y = {np.min(self.y_observed):.4f}")
        
        best_idx = np.argmin(self.y_observed)
        return self.X_observed[best_idx][0], self.y_observed[best_idx]

# Test function to optimize
def test_function(x):
    return x**2 + 10*np.sin(x)

# Run Bayesian Optimization
bo = BayesianOptimizationWithBias(test_function, bounds=(-5, 5), bias_strength=0.2)
best_x, best_y = bo.optimize(n_iter=30)

print(f"\nOptimization complete!")
print(f"Best x: {best_x:.4f}")
print(f"Best y: {best_y:.4f}")

## Visualization

In [ ]:
# Plot optimization results
plt.figure(figsize=(12, 4))

# Function and observations
plt.subplot(1, 3, 1)
x_range = np.linspace(-5, 5, 200)
y_true = test_function(x_range)
plt.plot(x_range, y_true, 'b-', label='True function')
plt.scatter(bo.X_observed.flatten(), bo.y_observed, c='red', s=50, label='Observations')
plt.scatter(best_x, best_y, c='green', s=100, marker='*', label='Best found')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Optimization Progress')
plt.legend()
plt.grid(True)

# Convergence plot
plt.subplot(1, 3, 2)
best_so_far = np.minimum.accumulate(bo.y_observed)
plt.plot(best_so_far, 'g-')
plt.xlabel('Iteration')
plt.ylabel('Best value found')
plt.title('Convergence')
plt.grid(True)

# Acquisition landscape
plt.subplot(1, 3, 3)
bo.gp.fit(bo.X_observed, bo.y_observed)
X_test = np.linspace(-5, 5, 200).reshape(-1, 1)
mu, sigma = bo.gp.predict(X_test)
ei = expected_improvement(mu, sigma, np.min(bo.y_observed))
plt.plot(X_test.flatten(), ei, 'purple')
plt.xlabel('x')
plt.ylabel('Expected Improvement')
plt.title('Acquisition Function')
plt.grid(True)

plt.tight_layout()
plt.show()

print("Visualization complete")

## Summary

Bayesian Optimization with bias:
- Efficient for expensive function optimization
- Uses Gaussian Process for surrogate modeling
- Acquisition functions guide sample selection
- Bias guides search toward promising regions

## Next Tutorial

Next: [Variable Neighborhood Search](06_VNS_Variable_Neighborhood_Search.ipynb)